In [2]:
"""
evaluate_validation_set.py — Full Dataset Evaluator

This script runs your predictor.py against an entire validation dataset
and calculates the final Macro F1 and per-class mIoU.
"""

import os
import sys
import json
import importlib.util
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
from PIL import Image, ImageDraw
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import torch

# =====================================================================
# CONFIGURATION
# =====================================================================
# Update these paths if your validation dataset is located elsewhere
VAL_IMG_DIR = Path("../../validation/validation/image").resolve()
VAL_ANNO_DIR = Path("../../validation/validation/annos").resolve()
BATCH_SIZE = 8 # Adjust based on your GPU VRAM (4 or 8 is safe)

# DeepFashion2 category_id → name
DEEPFASHION_CATID_TO_NAME: Dict[int, str] = {
    1: "short sleeve top", 2: "long sleeve top", 3: "short sleeve outwear",
    4: "long sleeve outwear", 5: "vest", 6: "sling", 7: "shorts",
    8: "trousers", 9: "skirt", 10: "short sleeve dress",
    11: "long sleeve dress", 12: "vest dress", 13: "sling dress",
}

def load_annotation(anno_path: Path, canonical_map: Dict[str, int]) -> List[Dict[str, Any]]:
    """Parse annotation JSON -> list of GT items (only canonical classes)."""
    with open(anno_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    items = []
    
    # Create reverse map for GT IDs
    catid_to_canonical = {}
    for cat_id, cat_name in DEEPFASHION_CATID_TO_NAME.items():
        if cat_name in canonical_map:
            catid_to_canonical[cat_id] = canonical_map[cat_name]

    for val in data.values():
        if not isinstance(val, dict) or "bounding_box" not in val:
            continue
        cat_id = val["category_id"]
        if cat_id not in catid_to_canonical:
            continue
        items.append({
            "segmentation": val["segmentation"],
            "canonical_idx": catid_to_canonical[cat_id],
        })
    return items

def rasterize_polygons(segmentation: list, width: int, height: int) -> np.ndarray:
    """Render polygon coordinate lists into a binary (H, W) mask."""
    canvas = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(canvas)
    for poly in segmentation:
        coords = [(poly[i], poly[i + 1]) for i in range(0, len(poly) - 1, 2)]
        if len(coords) >= 3:
            draw.polygon(coords, fill=1)
    return np.array(canvas, dtype=np.uint8)

def build_remap(student_mapping: dict, canonical_names: List[str]) -> Dict[int, int]:
    """Map student class index -> canonical class index."""
    name_to_idx = {name: i for i, name in enumerate(canonical_names)}
    remap = {}
    for s_idx, s_name in student_mapping.items():
        name = str(s_name).strip().lower()
        if name in name_to_idx:
            remap[int(s_idx)] = name_to_idx[name]
    return remap

def main():
    folder = Path.cwd()  
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🚀 Starting Full Validation on {device.upper()}")
    
    if not VAL_IMG_DIR.exists() or not VAL_ANNO_DIR.exists():
        print(f"❌ Error: Cannot find validation directories!")
        print(f"Images: {VAL_IMG_DIR}")
        print(f"Annos: {VAL_ANNO_DIR}")
        sys.exit(1)

    # 1. Import predictor.py
    try:
        spec = importlib.util.spec_from_file_location("predictor", folder / "predictor.py")
        predictor = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(predictor)
    except Exception as e:
        print(f"❌ Error importing predictor.py: {e}")
        sys.exit(1)

    # Extract canonical classes
    cls_names = [str(name).strip().lower() for name in predictor.CLS_CLASS_MAPPING.values() if str(name).strip().lower() != "background"]
    CANONICAL_CLASSES = cls_names
    NUM_CLASSES = len(CANONICAL_CLASSES)
    canonical_map = {name: i for i, name in enumerate(CANONICAL_CLASSES)}

    # Build remaps
    remap_cls = build_remap(predictor.CLS_CLASS_MAPPING, CANONICAL_CLASSES)
    remap_seg = build_remap(predictor.SEG_CLASS_MAPPING, CANONICAL_CLASSES)

    # Load images
    image_paths = sorted(list(VAL_IMG_DIR.glob("*.jpg")))
    print(f"📦 Found {len(image_paths)} images in validation set.")

    # =================================================================
    # CLASSIFICATION EVALUATION
    # =================================================================
    print("\n" + "="*50)
    print(" 1. EVALUATING CLASSIFICATION (Macro F1)")
    print("="*50)
    
    cls_model = predictor.load_classification_model(str(folder), device)
    
    all_gt_cls = []
    all_pred_cls = []
    
    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Classification"):
        batch_paths = image_paths[i:i+BATCH_SIZE]
        batch_imgs = [Image.open(p).convert("RGB") for p in batch_paths]
        
        # Get Predictions
        cls_outs = predictor.predict_classification(cls_model, batch_imgs)
        
        # Process GT and Predictions
        for p, out in zip(batch_paths, cls_outs):
            anno_path = VAL_ANNO_DIR / f"{p.stem}.json"
            if not anno_path.exists(): continue
            
            # GT
            gt_items = load_annotation(anno_path, canonical_map)
            gt_vec = np.zeros(NUM_CLASSES, dtype=np.int32)
            for item in gt_items:
                gt_vec[item["canonical_idx"]] = 1
            all_gt_cls.append(gt_vec)
            
            # Pred
            pred_vec = np.zeros(NUM_CLASSES, dtype=np.int32)
            for s_idx, val in enumerate(out["labels"]):
                if s_idx in remap_cls:
                    pred_vec[remap_cls[s_idx]] = val
            all_pred_cls.append(pred_vec)

    if all_gt_cls:
        y_true = np.vstack(all_gt_cls)
        y_pred = np.vstack(all_pred_cls)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0.0)
        print(f"\n✅ FINAL CLASSIFICATION MACRO F1: {macro_f1:.4f}\n")
    
    # Free VRAM
    del cls_model
    torch.cuda.empty_cache()


    # =================================================================
    # SEGMENTATION EVALUATION
    # =================================================================
    print("="*50)
    print(" 2. EVALUATING SEGMENTATION (mIoU)")
    print("="*50)
    
    det_model = predictor.load_detection_model(str(folder), device)
    
    total_intersection = np.zeros(NUM_CLASSES, dtype=np.float64)
    total_union = np.zeros(NUM_CLASSES, dtype=np.float64)
    IGNORE_LABEL = 255
    
    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Segmentation"):
        batch_paths = image_paths[i:i+BATCH_SIZE]
        batch_imgs = [Image.open(p).convert("RGB") for p in batch_paths]
        
        det_outs = predictor.predict_detection_segmentation(det_model, batch_imgs)
        
        for p, img, pred in zip(batch_paths, batch_imgs, det_outs):
            anno_path = VAL_ANNO_DIR / f"{p.stem}.json"
            if not anno_path.exists(): continue
            
            img_w, img_h = img.size
            gt_items = load_annotation(anno_path, canonical_map)
            
            # Build GT Map
            gt_sem = np.full((img_h, img_w), IGNORE_LABEL, dtype=np.uint8)
            for item in gt_items:
                gt_mask = rasterize_polygons(item["segmentation"], img_w, img_h)
                gt_sem[gt_mask == 1] = item["canonical_idx"]
                
            # Build Pred Map
            pred_sem = np.full((img_h, img_w), IGNORE_LABEL, dtype=np.uint8)
            pred_conf = np.full((img_h, img_w), -1.0, dtype=np.float32)
            
            for mask, score, label in zip(pred["masks"], pred["scores"], pred["labels"]):
                canonical = remap_seg.get(label)
                if canonical is None: continue
                
                binary = np.asarray(mask, dtype=np.uint8)
                
                # Auto-resize if student didn't format it right
                if binary.shape != (img_h, img_w):
                    mask_pil = Image.fromarray(binary * 255).resize((img_w, img_h), Image.NEAREST)
                    binary = (np.array(mask_pil) > 127).astype(np.uint8)
                
                higher = (binary == 1) & (score > pred_conf)
                pred_sem[higher] = canonical
                pred_conf[higher] = score
                
            # Accumulate Intersections and Unions
            for c in range(NUM_CLASSES):
                pred_c = (pred_sem == c)
                gt_c = (gt_sem == c)
                total_intersection[c] += np.logical_and(pred_c, gt_c).sum()
                total_union[c] += np.logical_or(pred_c, gt_c).sum()

    # Calculate final mIoU
    per_class_iou = []
    print("\n--- Per-Class IoU ---")
    for c in range(NUM_CLASSES):
        if total_union[c] > 0:
            iou = total_intersection[c] / total_union[c]
            per_class_iou.append(iou)
            print(f"[{CANONICAL_CLASSES[c]:<20}] IoU: {iou:.4f}")
        else:
            print(f"[{CANONICAL_CLASSES[c]:<20}] IoU: N/A")

    if per_class_iou:
        final_miou = np.mean(per_class_iou)
        print(f"\n✅ FINAL SEGMENTATION mIoU: {final_miou:.4f}")

if __name__ == "__main__":
    main()

🚀 Starting Full Validation on CUDA
📦 Found 32153 images in validation set.

 1. EVALUATING CLASSIFICATION (Macro F1)


Classification: 100%|██████████| 4020/4020 [17:29<00:00,  3.83it/s]



✅ FINAL CLASSIFICATION MACRO F1: 0.7336

 2. EVALUATING SEGMENTATION (mIoU)


Segmentation: 100%|██████████| 4020/4020 [1:01:50<00:00,  1.08it/s]



--- Per-Class IoU ---
[short sleeve top    ] IoU: 0.4216
[trousers            ] IoU: 0.2973
[shorts              ] IoU: 0.3492
[long sleeve top     ] IoU: 0.2402
[skirt               ] IoU: 0.2263

✅ FINAL SEGMENTATION mIoU: 0.3069
